# ENEM Profile Map — Cluster visualization

Gold inputs: `participant_embeddings`, `participant_clusters`, `cluster_evolution_uf_ano`.
Outputs: `figures/cluster_map.png`, `figures/cluster_evolution.png`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

DIR_GOLD = ROOT / "data" / "gold"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PATH_EMBEDDINGS = DIR_GOLD / "participant_embeddings.parquet"
PATH_CLUSTERS = DIR_GOLD / "participant_clusters.parquet"
PATH_EVOLUTION = DIR_GOLD / "cluster_evolution_uf_ano.parquet"
MAX_POINTS_MAP = 80_000
SEED = 42

## 1) Cluster Map — 2D reduction (PCA) colored by cluster

In [ ]:
emb = pd.read_parquet(PATH_EMBEDDINGS)
clu = pd.read_parquet(PATH_CLUSTERS)
df = emb.merge(clu, on=["nu_inscricao", "ano"], how="inner")
vecs = np.array(df["embedding_vector"].tolist(), dtype=np.float64)
labels = df["cluster_id"].values

if len(df) > MAX_POINTS_MAP:
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(df), MAX_POINTS_MAP, replace=False)
    vecs = vecs[idx]
    labels = labels[idx]

pca = PCA(n_components=2, random_state=SEED)
xy = pca.fit_transform(vecs)
var_ratio = pca.explained_variance_ratio_.sum()
print(f"PCA explained variance (2 components): {var_ratio:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="tab10", alpha=0.5, s=8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("ENEM Profile Map")
plt.colorbar(sc, ax=ax, label="Cluster")
ax.set_axisbelow(True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "cluster_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 2) Evolution 2020 → 2024 — % participants per cluster by UF

In [ ]:
ev = pd.read_parquet(PATH_EVOLUTION)
ev["ano"] = ev["ano"].astype(int)
anos = sorted(ev["ano"].unique())
ufs = sorted(ev["uf"].dropna().unique().tolist())
n_ufs = len(ufs)
print(f"UFs: {n_ufs}, Years: {anos}")

In [ ]:
# Biggest change per UF: cluster with max |pct_2024 - pct_2020|
wide = ev.pivot_table(index=["uf", "ano"], columns="cluster_id", values="pct_participants", aggfunc="first").reset_index()
pct_2020 = wide[wide["ano"] == 2020].set_index("uf") if 2020 in wide["ano"].values else None
pct_2024 = wide[wide["ano"] == 2024].set_index("uf") if 2024 in wide["ano"].values else None

biggest_change = {}
if pct_2020 is not None and pct_2024 is not None:
    for uf in ufs:
        if uf not in pct_2020.index or uf not in pct_2024.index:
            continue
        r0 = pct_2020.loc[uf]
        r4 = pct_2024.loc[uf]
        cluster_cols = [c for c in wide.columns if c not in ("uf", "ano") and isinstance(c, (int, np.integer))]
        deltas = [(c, abs(r4.get(c, 0) - r0.get(c, 0))) for c in cluster_cols]
        if deltas:
            c_max = max(deltas, key=lambda x: x[1])
            biggest_change[uf] = c_max[0]

In [ ]:
if n_ufs == 0:
    raise ValueError("No UFs in cluster_evolution_uf_ano; run cluster_profiles.py first.")
n_cols = 4
n_rows = max(1, (n_ufs + n_cols - 1) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = np.atleast_2d(axes)
cluster_ids = sorted(ev["cluster_id"].unique().tolist())
colors = plt.cm.tab10(np.linspace(0, 1, max(len(cluster_ids), 10)))

for i, uf in enumerate(ufs):
    ax = axes.flat[i]
    sub = ev[ev["uf"] == uf]
    for j, cid in enumerate(cluster_ids):
        line = sub[sub["cluster_id"] == cid].sort_values("ano")
        if line.empty:
            continue
        lw = 2.5 if biggest_change.get(uf) == cid else 1
        ax.plot(line["ano"], line["pct_participants"] * 100, label=f"C{cid}", color=colors[j % 10], linewidth=lw)
    ax.set_title(uf)
    ax.set_xlabel("Ano")
    ax.set_ylabel("% participants")
    ax.legend(loc="best", fontsize=6)
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

for j in range(len(ufs), axes.size):
    axes.flat[j].set_visible(False)
plt.suptitle("Cluster evolution by UF (2020 → 2024). Thick line = biggest % change.", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "cluster_evolution.png", dpi=150, bbox_inches="tight")
plt.show()